# Re-slotting: swaps, clusters and a shift simulation

This notebook does three things, in order.

1. **Swaps.** Rank pick-face locations by how much extra walking they cause, and swap the worst ones with quiet spots near the front of the aisle, but only when the walking saved is worth more than the cost of moving the product.
2. **Clusters.** Use Amelia's clusters (products that are ordered together) to move leftover misplaced products into their cluster's area. The busiest cluster gets the area closest to the dispatch board.
3. **Simulation.** Run the test month as a day at the warehouse, with pickers on shift, walking, picking and waiting for refills. Do it for three layouts (no change, swaps, swaps then clusters) under different conditions, and compare the labor cost.

Step 1 and step 2 are separate. Step 2 only touches locations that step 1 left alone.

In [ ]:
TRAVEL_DIR = "../data/processed/travel"
PICKS_FILE = "../data/raw/picks_sep2025_sep2026.csv"
INVENTORY_FILE = "../data/raw/inventory_2026-10-20.csv"
CLUSTER_FILE = "../data/processed/sku_clusters.csv"

TRAIN_MONTH = "2026-07"
TEST_MONTH = "2026-08"
DOCK = "D_183183.9_5933.3"

GOLDEN_FT = 60
MAX_SWAPS = 150
PARTNERS_TRIED = 5
COST_PER_CASE_MOVED = 0.17
WAGE_PER_HOUR = 45.50
WALKING_MPH = 1.25
DOLLARS_PER_FOOT = WAGE_PER_HOUR / WALKING_MPH / 5280

SECONDS_PER_PICK = 77.0
SECONDS_PER_FOOT = 0.377
FACE_FRACTION_OF_PALLET = 0.75
REPLICATIONS = 5

## Load the data

- **Travel network** (Amelia): the walking distance in feet between every pair of stops, and which stop each pick-face location sits at.
- **Picks**: one row per pick. A **tour** is one operator's picks for one order, with at least 3 picks.
- **Inventory**: cases sitting in each location, used for the move cost.
- **Clusters** (Amelia): which SKUs belong to which cluster.

In [ ]:
import zlib, heapq, itertools, collections
import numpy as np
import pandas as pd
from scipy import stats

distances = pd.read_csv(TRAVEL_DIR + "/stop_distance_ft.csv", index_col=0)
table = distances.values
row_number = {stop: i for i, stop in enumerate(distances.index)}
def feet(stop_a, stop_b):
    return table[row_number[stop_a], row_number[stop_b]]

crosswalk = pd.read_csv(TRAVEL_DIR + "/location_stop_crosswalk.csv", dtype=str)
crosswalk = crosswalk[(crosswalk["status"] == "ok") & (crosswalk["zone"] == "WZPCK") & (crosswalk["slot_type"] == "F")]
stop_of = dict(zip(crosswalk["location"], crosswalk["stop_id"]))
aisle_of = dict(zip(crosswalk["location"], crosswalk["aisle"]))

all_picks = pd.read_csv(PICKS_FILE, encoding="latin-1", dtype=str,
                        usecols=["NO", "CREATE_DT", "ORD_NO", "USER_NAME", "FROM_LOCATION", "FROM_UOM",
                                 "FROM_ZONE", "FROM_ITEM", "FROM_QTY", "CASES_PER_PALLET"])
all_picks = all_picks[(all_picks["FROM_UOM"] == "CS") & (all_picks["FROM_ZONE"] == "WZPCK")].copy()
all_picks["time"] = pd.to_datetime(all_picks["CREATE_DT"])
all_picks["month"] = all_picks["time"].dt.strftime("%Y-%m")

picks = all_picks[all_picks["FROM_LOCATION"].isin(stop_of)].copy()
picks["tour"] = picks["ORD_NO"] + " | " + picks["USER_NAME"]
picks = picks[picks.groupby("tour")["NO"].transform("size") >= 3]
picks["qty"] = picks["FROM_QTY"].astype(float)

def get_tours(month):
    rows = picks[picks["month"] == month].sort_values(["tour", "time"])
    return {tour: list(r["FROM_LOCATION"]) for tour, r in rows.groupby("tour")}

train_tours = get_tours(TRAIN_MONTH)
test_tours = get_tours(TEST_MONTH)
print(len(train_tours), "training tours,", len(test_tours), "test tours")

inventory = pd.read_csv(INVENTORY_FILE, dtype=str)
inventory = inventory[inventory["UOM"] == "CS"]
cases_on_hand = inventory.assign(QTY=inventory["QTY"].astype(float)).groupby("LOCATION")["QTY"].sum().to_dict()
typical_cases = float(np.median(list(cases_on_hand.values())))
def cases_in(location):
    return cases_on_hand.get(location, typical_cases)

sku_clusters = pd.read_csv(CLUSTER_FILE, dtype={"sku": str})
cluster_of_sku = dict(zip(sku_clusters["sku"], sku_clusters["cluster"]))
print(len(cluster_of_sku), "SKUs in", sku_clusters["cluster"].nunique(), "clusters")

## Walking helpers and the golden zone

- `tour_length` is the feet walked for a tour: dock, each stop in order, back to the dock.
- `replan` rebuilds a tour after swaps. A stop that moved is put back in the cheapest place in the route. The other stops keep their order.
- The **golden zone** is any location within 60 ft of the front of its aisle (the end closest to the dispatch board).

In [ ]:
def tour_length(locations):
    stops = [DOCK] + [stop_of[loc] for loc in locations] + [DOCK]
    return sum(feet(stops[i], stops[i + 1]) for i in range(len(stops) - 1))

def replan(locations, moved_to):
    route = [loc for loc in locations if loc not in moved_to]
    for loc in locations:
        if loc in moved_to:
            new_stop = stop_of[moved_to[loc]]
            def feet_added(position):
                before = stop_of[route[position - 1]] if position > 0 else DOCK
                after = stop_of[route[position]] if position < len(route) else DOCK
                return feet(before, new_stop) + feet(new_stop, after) - feet(before, after)
            route.insert(min(range(len(route) + 1), key=feet_added), moved_to[loc])
    return route

feet_from_dock = {loc: feet(DOCK, stop) for loc, stop in stop_of.items()}
aisle_front = {}
for loc, aisle in aisle_of.items():
    aisle_front[aisle] = min(aisle_front.get(aisle, 1e12), feet_from_dock[loc])
golden = {loc for loc in stop_of if feet_from_dock[loc] - aisle_front[aisle_of[loc]] <= GOLDEN_FT}
print(len(golden), "golden locations out of", len(stop_of))

## Step 1: swaps

**Detour** is the extra walking a stop adds to a tour: (stop before to this stop) + (this stop to the stop after) - (stop before to stop after). Added up over every visit in the training month, it measures how costly a location is. A busy location counts more because it is visited more often, so velocity is already in the number.

For each costly location, starting with the worst, try the 5 quietest unused golden locations as swap partners. For each one:
- walking saved = feet saved on the tours that touch either location, at $0.0069 per foot
- move cost = $0.17 per case, for the cases in both locations

Keep the partner with the best saving minus cost, and only if the saving is at least the cost.

In [ ]:
def detour_ranking(tours):
    detour_total, visit_count = {}, {}
    for locations in tours.values():
        locations = [loc for i, loc in enumerate(locations) if i == 0 or loc != locations[i - 1]]
        stops = [DOCK] + [stop_of[loc] for loc in locations] + [DOCK]
        for i, loc in enumerate(locations, start=1):
            detour = feet(stops[i - 1], stops[i]) + feet(stops[i], stops[i + 1]) - feet(stops[i - 1], stops[i + 1])
            detour_total[loc] = detour_total.get(loc, 0) + max(detour, 0)
            visit_count[loc] = visit_count.get(loc, 0) + 1
    return pd.DataFrame({"visits": visit_count, "detour_ft": detour_total}).sort_values("detour_ft", ascending=False)

ranking = detour_ranking(train_tours)
visits = ranking["visits"].to_dict()
tours_through = {}
for tour, locations in train_tours.items():
    for loc in set(locations):
        tours_through.setdefault(loc, []).append(tour)

def try_swap(costly, partner, moved_to):
    touched = set(tours_through.get(costly, [])) | set(tours_through.get(partner, []))
    with_swap = {**moved_to, costly: partner, partner: costly}
    feet_before = sum(tour_length(replan(train_tours[t], moved_to)) for t in touched)
    feet_after = sum(tour_length(replan(train_tours[t], with_swap)) for t in touched)
    saved = (feet_before - feet_after) * DOLLARS_PER_FOOT
    cases = cases_in(costly) + cases_in(partner)
    return {"saved": saved, "cost": COST_PER_CASE_MOVED * cases, "cases": cases, "feet": feet_before - feet_after, "with_swap": with_swap}

def run_swaps(costly_locations, partners_for, moved_to, step):
    moved_to, log = dict(moved_to), []
    for costly in costly_locations:
        if len(log) == MAX_SWAPS:
            break
        if costly in moved_to:
            continue
        best = None
        for partner in partners_for(costly, moved_to)[:PARTNERS_TRIED]:
            trial = try_swap(costly, partner, moved_to)
            if trial["saved"] >= trial["cost"] and (best is None or trial["saved"] - trial["cost"] > best["saved"] - best["cost"]):
                best = {**trial, "partner": partner}
        if best:
            moved_to = best["with_swap"]
            log.append({"step": step, "costly location": costly, "moves to": best["partner"],
                        "visits": visits.get(costly, 0), "detour ft (month)": round(ranking.loc[costly, "detour_ft"]) if costly in ranking.index else 0,
                        "feet saved": round(best["feet"]), "$ saved": round(best["saved"], 2),
                        "cases moved": int(best["cases"]), "$ move cost": round(best["cost"], 2)})
    return moved_to, pd.DataFrame(log)

def quietest_golden(costly, moved_to):
    if costly in golden:
        return []
    return sorted((loc for loc in golden if loc in visits and loc not in moved_to),
                  key=lambda loc: (visits[loc], cases_in(loc), feet_from_dock[loc]))

swapped_to, swap_log = run_swaps(list(ranking.index), quietest_golden, {}, "1 swaps")
print(len(swap_log), "swaps, move cost $", round(swap_log["$ move cost"].sum()))
swap_log.head(10)

## Step 2: clusters

1. Each location is given a cluster: the cluster of the product most often picked there in the training month.
2. Clusters are ranked busiest first (average visits per location). The locations that belong to clusters are sorted by distance from the dispatch board and handed out in blocks: the busiest cluster gets the closest block, and so on.
3. A location whose cluster does not match the block it sits in is misplaced. Try to swap it with one of the 5 quietest locations in its own cluster's block, using the same saving-versus-cost test as step 1.

Step 2 starts from the layout after step 1, and locations step 1 moved are left alone.

In [ ]:
train_items = picks[picks["month"] == TRAIN_MONTH].groupby("FROM_LOCATION")["FROM_ITEM"].agg(lambda s: s.value_counts().index[0])
loc_cluster = {loc: cluster_of_sku[item] for loc, item in train_items.items() if item in cluster_of_sku}

visits_in_cluster = {}
for loc, c in loc_cluster.items():
    visits_in_cluster.setdefault(c, []).append(visits.get(loc, 0))
busiest_first = sorted(visits_in_cluster, key=lambda c: -np.mean(visits_in_cluster[c]))
closest_first = sorted(loc_cluster, key=lambda loc: feet_from_dock[loc])
block, start = {}, 0
for c in busiest_first:
    block[c] = set(closest_first[start:start + len(visits_in_cluster[c])])
    start += len(visits_in_cluster[c])

misplaced = [loc for loc in ranking.index if loc in loc_cluster and loc not in block[loc_cluster[loc]]]

def own_cluster_spots(costly, moved_to):
    c = loc_cluster[costly]
    return sorted((loc for loc in block[c] if loc_cluster.get(loc) != c and loc not in moved_to),
                  key=lambda loc: (visits.get(loc, 0), cases_in(loc)))

clustered_to, cluster_log = run_swaps(misplaced, own_cluster_spots, swapped_to, "2 clusters")
print(len(cluster_log), "extra swaps from clusters, move cost $", round(cluster_log["$ move cost"].sum()))
pd.DataFrame({"cluster": busiest_first, "locations": [len(block[c]) for c in busiest_first],
              "avg visits per location": [round(np.mean(visits_in_cluster[c]), 1) for c in busiest_first]})

## Replay: how much walking each plan saves

The plans were built from the training month. Here the test month's real tours are walked on each layout and the feet are added up. Walking is valued at $0.0069 per foot (1.25 mph), and the move cost is subtracted.

In [ ]:
def replay(moved_to, move_cost):
    before = sum(tour_length(t) for t in test_tours.values())
    after = sum(tour_length(replan(t, moved_to)) for t in test_tours.values()) if moved_to else before
    saved = (before - after) * DOLLARS_PER_FOOT
    return {"swaps": len(moved_to) // 2, "walking saved %": round(100 * (before - after) / before, 1),
            "$ saved": round(saved), "$ move cost": round(move_cost), "$ net": round(saved - move_cost)}

swap_cost = swap_log["$ move cost"].sum()
all_cost = swap_cost + cluster_log["$ move cost"].sum()
pd.DataFrame({"1 no change": replay({}, 0), "2 swaps": replay(swapped_to, swap_cost),
              "3 swaps then clusters": replay(clustered_to, all_cost)}).T

## The simulation

The replay only counts feet. The simulation plays the test month as a series of events in clock order, so waiting and refilling matter too.

**What happens in the simulation**
- **Pickers.** The number on shift each hour comes from the real crew pattern in the test month. A tour waits in a queue until a picker is free.
- **Tour time.** 77 seconds per pick plus 0.377 seconds per foot walked. The feet come from the layout being tested.
- **Stock.** Each pick face holds 75% of a pallet of its product when full, and starts the month 30 to 100% full at random. Stock follows the product when it is swapped.
- **Refills.** When a face cannot cover a pick, a replenisher is called. There are two, each refill takes 6 minutes, and requests are served in order. The picker waits for it.
- **Labor cost.** (time pickers spend working + time waiting + time replenishers spend refilling) x $45.50 per hour.

**Variability.** Each day's number of tours is drawn with about 10% random variation, tours start a few minutes earlier or later, and start stock is random. Every layout is run with the same random draws, so differences between layouts are not noise. Each case is repeated 5 times.

**Calibration.** The face size gives about 226 refills a day with no change, close to the 238 refill transactions a day in the real data.

In [ ]:
cases_per_pallet = pd.to_numeric(all_picks["CASES_PER_PALLET"], errors="coerce")
pallet_of = all_picks.assign(cpp=cases_per_pallet).query("cpp > 0").groupby("FROM_LOCATION")["cpp"].median().to_dict()
typical_pallet = float(cases_per_pallet[cases_per_pallet > 0].median())

month_picks = all_picks[all_picks["month"] == TEST_MONTH]
month_picks = month_picks.assign(hour=month_picks["time"].dt.hour, day=month_picks["time"].dt.date,
                                 weekend=(month_picks["time"].dt.dayofweek >= 5).astype(int))
pickers_per_hour = month_picks.groupby(["weekend", "day", "hour"])["USER_NAME"].nunique().groupby(level=[0, 2]).mean()
crew = {w: [float(pickers_per_hour.get((w, h), 1.0)) for h in range(24)] for w in (0, 1)}

test_rows = picks[picks["month"] == TEST_MONTH].sort_values(["tour", "time"])
test_days = collections.defaultdict(list)
for tour, rows in test_rows.groupby("tour"):
    first = rows["time"].iloc[0]
    test_days[first.strftime("%Y-%m-%d")].append((first.hour * 3600 + first.minute * 60 + first.second,
                                                  list(rows["FROM_LOCATION"]), list(rows["qty"])))
test_days = dict(sorted(test_days.items()))
tour_share = sum(len(t[1]) for d in test_days.values() for t in d) / len(month_picks)
print(len(test_days), "days,", round(100 * tour_share), "% of case picks are in tours that can be placed on the map")

def prepare(moved_to):
    days = {}
    for day, tours in test_days.items():
        planned = []
        for start, locations, qtys in tours:
            route = replan(locations, moved_to) if moved_to else locations
            by_location = collections.defaultdict(list)
            for loc, q in zip(locations, qtys):
                by_location[moved_to.get(loc, loc)].append(q)
            stops = [DOCK] + [stop_of[loc] for loc in route] + [DOCK]
            legs = [feet(stops[i], stops[i + 1]) for i in range(len(stops) - 1)]
            planned.append((start, route, [by_location[loc].pop(0) for loc in route], legs))
        days[day] = planned
    return days

In [ ]:
def simulate(planned_days, moved_to, seed, staff=1.0, demand=1.0, replenishers=2, refill_minutes=6.0,
             face_fraction=FACE_FRACTION_OF_PALLET, seconds_per_foot=SECONDS_PER_FOOT):
    rng = np.random.default_rng(seed)
    product_at = lambda face: moved_to.get(face, face)
    face_size = {}
    def size(face):
        if face not in face_size:
            face_size[face] = pallet_of.get(product_at(face), typical_pallet) * face_fraction
        return face_size[face]
    stock = {}
    def start_stock(face):
        fill = np.random.default_rng(zlib.crc32(str(product_at(face)).encode()) + seed).uniform(0.3, 1.0)
        return fill * size(face)

    total = dict(feet=0.0, working_s=0.0, waiting_s=0.0, replenisher_s=0.0, refills=0)
    counter = itertools.count()
    refill_seconds = refill_minutes * 60

    for day, tours in planned_days.items():
        weekend = 1 if pd.Timestamp(day).dayofweek >= 5 else 0
        count = int(round(len(tours) * demand * max(rng.normal(1.0, 0.10), 0.3)))
        chosen = rng.integers(0, len(tours), count)
        jitter = rng.uniform(-1800, 1800, count)
        events, waiting_for_refill, replenisher_free = [], {}, [0.0] * replenishers
        queue, pickers_busy = collections.deque(), [0]

        def pickers_on_shift(t):
            return max(1, min(40, int(round(crew[weekend][min(int(t // 3600), 23)] * tour_share * staff))))

        for k, i in enumerate(chosen):
            arrive = float(min(max(tours[i][0] + jitter[k], 0), 86399))
            heapq.heappush(events, (arrive, next(counter), "arrive", tours[i]))
        for hour in range(25):
            heapq.heappush(events, (hour * 3600.0, next(counter), "hour", None))

        def call_replenisher(face, t):
            if face not in waiting_for_refill:
                r = min(range(replenishers), key=lambda q: replenisher_free[q])
                done = max(t, replenisher_free[r]) + refill_seconds
                replenisher_free[r] = done
                waiting_for_refill[face] = done
                total["replenisher_s"] += refill_seconds
                total["refills"] += 1
            return waiting_for_refill[face]

        def start_tours(t):
            while queue and pickers_busy[0] < pickers_on_shift(t):
                tour = queue.popleft()
                pickers_busy[0] += 1
                tour["began"], tour["waited"] = t, 0.0
                heapq.heappush(events, (t + tour["legs"][0] * seconds_per_foot, next(counter), "at stop", tour))

        def next_event(tour, t):
            tour["j"] += 1
            t_done = t + SECONDS_PER_PICK
            if tour["j"] < len(tour["route"]):
                heapq.heappush(events, (t_done + tour["legs"][tour["j"]] * seconds_per_foot, next(counter), "at stop", tour))
            else:
                heapq.heappush(events, (t_done + tour["legs"][-1] * seconds_per_foot, next(counter), "finish", tour))

        while events:
            t, _, kind, tour = heapq.heappop(events)
            if kind == "arrive":
                _, route, qtys, legs = tour
                queue.append({"route": route, "qtys": qtys, "legs": legs, "j": 0})
                start_tours(t)
            elif kind == "hour":
                start_tours(t)
            elif kind == "at stop":
                j = tour["j"]
                face, q = tour["route"][j], tour["qtys"][j]
                if face in waiting_for_refill and waiting_for_refill[face] <= t:
                    stock[face] = size(face)
                    del waiting_for_refill[face]
                if face not in stock:
                    stock[face] = start_stock(face)
                if stock[face] < q:
                    done = call_replenisher(face, t)
                    tour["waited"] += max(done - t, 0)
                    heapq.heappush(events, (max(done, t), next(counter), "refilled", tour))
                else:
                    stock[face] -= q
                    next_event(tour, t)
            elif kind == "refilled":
                face, q = tour["route"][tour["j"]], tour["qtys"][tour["j"]]
                if face in waiting_for_refill:
                    stock[face] = size(face)
                    del waiting_for_refill[face]
                stock[face] = max(stock[face], q) - q
                next_event(tour, t)
            elif kind == "finish":
                pickers_busy[0] -= 1
                total["feet"] += sum(tour["legs"])
                total["working_s"] += (t - tour["began"]) - tour["waited"]
                total["waiting_s"] += tour["waited"]
                start_tours(t)
        for face in list(waiting_for_refill):
            stock[face] = size(face)

    total["labor_$"] = (total["working_s"] + total["waiting_s"] + total["replenisher_s"]) / 3600 * WAGE_PER_HOUR
    return total

prepared = {"1 no change": prepare({}), "2 swaps": prepare(swapped_to), "3 swaps then clusters": prepare(clustered_to)}
layout_moves = {"1 no change": {}, "2 swaps": swapped_to, "3 swaps then clusters": clustered_to}
layout_cost = {"1 no change": 0.0, "2 swaps": swap_cost, "3 swaps then clusters": all_cost}
one_run = simulate(prepared["1 no change"], {}, seed=0)
print("no change, one run:", round(one_run["refills"] / len(test_days)), "refills a day, labor $", round(one_run["labor_$"]))

## Scenarios

Each scenario changes one thing and leaves the rest alone, so you can see what each condition does to the benefit.

| Scenario | What changes |
|---|---|
| Baseline | nothing |
| Fewer pickers | 25% fewer pickers on shift |
| More pickers | 25% more pickers on shift |
| Busier month | 20% more orders |
| Quieter month | 20% fewer orders |
| One replenisher | one instead of two |
| Slow refills | 12 minutes per refill instead of 6 |
| Smaller faces | each pick face holds half as much |
| Walking slower | 0.57 seconds per foot instead of 0.377 |
| Walking faster | 0.19 seconds per foot |

In [ ]:
scenarios = {
    "Baseline": {},
    "Fewer pickers (-25%)": {"staff": 0.75},
    "More pickers (+25%)": {"staff": 1.25},
    "Busier month (+20% orders)": {"demand": 1.2},
    "Quieter month (-20% orders)": {"demand": 0.8},
    "One replenisher": {"replenishers": 1},
    "Slow refills (12 min)": {"refill_minutes": 12.0},
    "Smaller faces (half size)": {"face_fraction": FACE_FRACTION_OF_PALLET / 2},
    "Walking slower (0.57 s/ft)": {"seconds_per_foot": 0.57},
    "Walking faster (0.19 s/ft)": {"seconds_per_foot": 0.19},
}

runs = []
for scenario, settings in scenarios.items():
    for layout in prepared:
        for rep in range(REPLICATIONS):
            result = simulate(prepared[layout], layout_moves[layout], rep, **settings)
            runs.append({"scenario": scenario, "layout": layout, "rep": rep, "labor": result["labor_$"], "refills": result["refills"]})
runs = pd.DataFrame(runs)

rows = []
for scenario in scenarios:
    s = runs[runs["scenario"] == scenario]
    base = s[s["layout"] == "1 no change"].set_index("rep")["labor"]
    for layout in ["2 swaps", "3 swaps then clusters"]:
        net = (base - s[s["layout"] == layout].set_index("rep")["labor"]) - layout_cost[layout]
        half_width = stats.t.ppf(0.975, len(net) - 1) * net.std(ddof=1) / np.sqrt(len(net))
        rows.append({"scenario": scenario, "layout": layout, "labor saved $": round(base.mean() - s[s["layout"] == layout]["labor"].mean()),
                     "move cost $": round(layout_cost[layout]), "net $": round(net.mean()),
                     "95% range": f"{net.mean() - half_width:,.0f} to {net.mean() + half_width:,.0f}"})
results = pd.DataFrame(rows)
results

## How to read the results

- **Net $** is the labor saved in the test month minus the cost of making the moves. It is a monthly figure that assumes the layout is re-planned each month.
- **95% range** comes from the 5 repetitions. If it crosses zero, the benefit in that scenario cannot be told apart from no benefit.
- Where refilling is the bottleneck (one replenisher, slow refills), the benefit shrinks, because pickers wait on refills and walking less does not help that.
- The walking-speed rows show the main assumption. The benefit depends on how long a foot of walking really takes.

## Assumptions and limits

| Assumption | Detail |
|---|---|
| Wage | $45.50 per hour all-in (confirmed range $45 to $46) |
| Move cost | $0.17 per case moved |
| Replay walking speed | 1.25 mph ($0.0069 per foot) |
| Simulation time per tour | 77 seconds per pick + 0.377 seconds per foot (fitted, not measured per picker) |
| Replenishers | 2, 6 minutes per refill, first come first served |
| Pick face size | 75% of a pallet of the product, calibrated to about 226 refills a day |
| Starting stock | random, 30 to 100% full |
| Golden zone | within 60 ft of the front of the aisle |
| Clusters | Amelia's file, built from all the data up to September 2026, so the cluster step has seen the test month. A true forward test would rebuild the clusters from data up to the training month. |
| Not included | aisle congestion, shelf height, the tall/short slot rule, pallet picks, picks outside the map, where trips end (all trips start and end at the dispatch board) |